In [1]:
# @title Importing

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.profiler import schedule
import torchmetrics
from ignite.engine import Events
from ignite.metrics import Metric, Loss
from ignite.handlers import global_step_from_engine, EarlyStopping
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau, CosineAnnealingWarmRestarts, ChainedScheduler
from ignite.contrib.handlers import TensorboardLogger
from ignite.contrib.handlers.tensorboard_logger import *
from ignite.exceptions import NotComputableError

import lightning as L
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, ModelSummary
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.profilers import SimpleProfiler, PyTorchProfiler
from lightning.pytorch.utilities.warnings import PossibleUserWarning

from torch.optim.lr_scheduler import SequentialLR

from captum.attr import Saliency, IntegratedGradients, Occlusion, GradientShap, ShapleyValueSampling, ShapleyValues, FeatureAblation, FeaturePermutation

import shap

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA, FastICA

import statsmodels.api as sm
from statsmodels.multivariate.factor_rotation import promax
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib import colors
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import colormaps as cmaps
import plotly.graph_objects as go

from tqdm.auto import tqdm

import seaborn as sns

from scipy import stats
from scipy.stats import norm, ttest_rel, ttest_1samp, wilcoxon, ttest_ind
import scipy.io as sio
from scipy.interpolate import griddata

from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import datetime
import time
import os
import random
import sys
import glob
import ipynbname
import pickle
import json
import math
import warnings
import copy
import logging
from pathlib import Path

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", category=PossibleUserWarning)
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=r".*Checkpoint directory .* exists and is not empty.*", category=UserWarning, module=r"lightning\.pytorch\.callbacks\.model_checkpoint")

SEED = 1

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    torch.set_float32_matmul_precision("highest")
    torch.use_deterministic_algorithms(True, warn_only=False)


seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\A\miniconda3\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
C:\Users\A\AppData\Local\Temp\ipykernel_3496\1352675481.py:17: DeprecationWarning: c:\Users\A\miniconda3\lib\site-packages\ignite\contrib\handlers\tensorboard_logger.py has been moved to /ignite/handlers/tensorboard_logger.py and will be removed in version 0.6.0.
 Please refer to the documentation for more details.
  from ignite.contrib.handlers.tensorboard_logger import *


In [2]:
# @title Configuration

class DotDict(dict):
    def __init__(self, d=None):
        super().__init__()
        if d:
            for k, v in d.items():
                self[k] = self._wrap(v)

    def _wrap(self, value):
        if isinstance(value, dict):
            return DotDict(value)
        if isinstance(value, list):
            return [self._wrap(v) for v in value]
        return value

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)

    def __setattr__(self, name, value):
        self[name] = self._wrap(value)

    def __delattr__(self, name):
        try:
            del self[name]
        except KeyError:
            raise AttributeError(name)

    def to_dict(self):
        out = {}
        for k, v in self.items():
            if isinstance(v, DotDict):
                out[k] = v.to_dict()
            elif isinstance(v, list):
                out[k] = [x.to_dict() if isinstance(x, DotDict) else x for x in v]
            else:
                out[k] = v
        return out

    def __deepcopy__(self, memo):
        return DotDict(copy.deepcopy(self.to_dict(), memo))


Conf = DotDict({
    "run_id": 1,
    "seed": SEED,
    "paths": {
        "data": "data.mat",
        "logs": "/lightning_logs",
    },
    "session_idx": 18,
    "device": device,
    "training": {
        "batch_size": 32,
        "max_epoch": 500,
        "min_delta": 1e-4,
        "patience": 10,
    },
    "model_type": {
        "name": "mean",
        "mean": {
            "n_hidden": 16,
            "n_heads": 2,
            "n_layers": 1,
            "nonlinearity": "gelu",
            "dropout": 0,
        },
        "cov": {
            "n_latent": 16,
            "n_hidden": 2,
            "n_heads": 1,
            "n_layers": 1,
            "nonlinearity": "gelu",
            "dropout": 0,
        },
    },
    "optimization": {
        "Adam": {
            "lr": 0.01,
            "weight_decay": 0,
        },
        "Reduce": {
            "factor": 0.5,
            "patience":5,
            "min_lr": 1e-10,
        },
    }
})

In [3]:
# @title Classes


class Anscombe(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 2.0 * torch.sqrt(x + 3.0 / 8.0)

    def inv(self, x):
        return (x / 2.0) ** 2 - 3.0 / 8.0

     
class NeuralDataset(Dataset):
    def __init__(self, x_position_vars, x_dense_vars, x_sparse_vars, Y, mean):
        self.x_position_vars = torch.tensor(x_position_vars, dtype=torch.float32).transpose(1, 2)
        self.x_dense_vars = torch.tensor(x_dense_vars, dtype=torch.float32)
        self.x_sparse_vars = torch.tensor(x_sparse_vars, dtype=torch.float32)
        
        Y_tensor = torch.tensor(Y, dtype=torch.float32).transpose(1, 2)
        anscombe = Anscombe()
        Y_anscombed = anscombe.forward(Y_tensor)
        self.Y = Y_anscombed - mean.cpu()
        
    def __len__(self):
        return self.x_position_vars.size(0)

    def __getitem__(self, idx):
        X = (self.x_position_vars[idx], self.x_dense_vars[idx], self.x_sparse_vars[idx])
        y = self.Y[idx]
        return X, y


class MVNNLLLoss(nn.Module):
    def __init__(self, reduction="mean"):
        super().__init__()

    def forward(self, out, y):
        mean, L = out
        
        if not torch.is_tensor(mean):
            mean = torch.stack(list(mean), dim=0)
        if not torch.is_tensor(y):
            y = torch.stack(list(y), dim=0)
        if not torch.is_tensor(L):
            L = torch.stack(list(L), dim=0)

        B = mean.shape[0]
        mean = mean.reshape(B, -1)
        y = y.reshape(B, -1)
        D = mean.shape[1]

        if L.dim() == 2:
            L = L.unsqueeze(0).expand(B, -1, -1)

        diff = (y - mean).unsqueeze(-1)
        sol = torch.linalg.solve_triangular(L, diff, upper=False).squeeze(-1)
        maha = (sol * sol).sum(dim=-1)
        logdet = 2.0 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(dim=-1)
        log_prob = -0.5 * (D * math.log(2.0 * math.pi) + logdet + maha)
        loss = (-log_prob / D)
        
        return loss


class Model(nn.Module):
    def __init__(self, Conf=None, **kwargs):
        super().__init__()
        self.n_position_vars = Conf.data.n_position_vars
        self.n_dense_vars = Conf.data.n_dense_vars
        self.n_sparse_vars = Conf.data.n_sparse_vars
        self.n_vars = Conf.data.n_vars
        self.n_bins = Conf.data.n_bins
        self.n_units = Conf.data.n_units
        self.device = Conf.device


class Time2Vec(nn.Module):
    def __init__(self, n_hidden):
        super().__init__()
        self.w0 = nn.Parameter(torch.randn(1))
        self.b0 = nn.Parameter(torch.randn(1))
        self.w = nn.Parameter(torch.randn(n_hidden - 1))
        self.b = nn.Parameter(torch.randn(n_hidden - 1))

    def forward(self, t):
        t = t.unsqueeze(-1)
        linear = self.w0 * t + self.b0
        periodic = torch.sin(self.w * t + self.b)
        return torch.cat([linear, periodic], dim=-1)


class Time2VecPositionalEncoding(nn.Module):
    def __init__(self, n_hidden):
        super().__init__()
        self.n_hidden = n_hidden
        self.t2v = Time2Vec(n_hidden)

    def forward(self, n_bins, device):
        t = torch.arange(n_bins, device=device).float()
        enc = self.t2v(t)
        return enc.unsqueeze(0)


class Transformer():
    def __init__(self, Conf=None, **kwargs):
        super().__init__(Conf=Conf, **kwargs)
        self.n_hidden = Conf.model_type[Conf.model_type.name].n_hidden
        self.n_heads = Conf.model_type[Conf.model_type.name].n_heads
        self.n_layers = Conf.model_type[Conf.model_type.name].n_layers
        self.nonlinearity = Conf.model_type[Conf.model_type.name].nonlinearity
        self.dropout = Conf.model_type[Conf.model_type.name].dropout

        self.vars_proj = nn.Linear(self.n_vars, self.n_hidden)
        
        self.positional_encoding = Time2VecPositionalEncoding(self.n_hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.n_hidden,
            nhead=self.n_heads,
            dim_feedforward=self.n_hidden * 4,
            dropout=self.dropout,
            activation=self.nonlinearity,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.n_layers)
        self.dropout_layer = nn.Dropout(self.dropout)
        

    def forward_transformer(self, x):
        x_position_vars, x_dense_vars, x_sparse_vars = x
        
        dense_vars = x_dense_vars.unsqueeze(1).expand(-1, self.n_bins, -1)
        sparse_vars = x_sparse_vars.unsqueeze(1).expand(-1, self.n_bins, -1)
        stacked = torch.cat([x_position_vars, dense_vars, sparse_vars], dim=2)
        stacked = self.vars_proj(stacked)
        
        pos_enc = self.positional_encoding(self.n_bins, stacked.device)
        combined = stacked + pos_enc
        transformed = self.transformer_encoder(combined)
        transformed = self.dropout_layer(transformed)
        return transformed


class MeanModel(Model):
    def __init__(self, Conf):
        super().__init__(Conf)


class ZeroMeanModel(MeanModel):
    def __init__(self, Conf):
        super().__init__(Conf)

    def forward(self, x):
        batchsize = x[0].size(0)
        return torch.zeros(batchsize, self.nbins, self.nunits, device=x[0].device, dtype=x[0].dtype)


class BaselineMeanModel(MeanModel):
    def __init__(self, Conf):
        super().__init__(Conf)
        self.theta = nn.Parameter(torch.empty(self.n_bins, self.n_units, device=self.device))

        with torch.no_grad():
            nn.init.normal_(self.theta, mean=0, std=1e-1)
            
    def forward(self, x):
        batch_size = x[0].size(0)
        mean = self.theta
        return mean.view(1, self.n_bins, self.n_units).expand(batch_size, -1, -1)


class ConditionalMeanModel(Transformer, MeanModel):
    def __init__(self, Conf):
        Conf_ = copy.deepcopy(Conf)
        Conf_.model_type.name = "mean"
        super().__init__(Conf=Conf_)

        self.output_size = self.n_units
        self.head = nn.Linear(self.n_hidden, self.output_size)

    def forward(self, x):
        transformed = self.forward_transformer(x)
        mean = self.head(transformed)
        return mean
             

class CovModel(Model):
    def __init__(self, Conf):
        super().__init__(Conf)
        self.n_latent = Conf.model_type.cov.n_latent
        
        self.register_buffer("I", torch.eye(self.n_bins * self.n_units, dtype=torch.float32, device=self.device))
        self.length_scales = nn.Parameter(torch.empty(self.n_latent, device=self.device))
        self.noise = nn.Parameter(torch.empty(self.n_bins, self.n_units, device=self.device))

        with torch.no_grad():
            ls0 = math.log(math.expm1(0.5))
            self.length_scales.fill_(ls0)
            nn.init.normal_(self.length_scales, mean=ls0, std=1)
            nn.init.normal_(self.noise, mean=-20.0, std=1)

    def kernel(self, fun, n_samples1, n_samples2, l):
        i_grid = torch.arange(n_samples1, dtype=torch.float32, device=l.device).view(-1, 1)
        j_grid = torch.arange(n_samples2, dtype=torch.float32, device=l.device).view(1, -1)
        return fun(i_grid, j_grid, l)

    def squared_exponential_kernel(self, x1, x2, l):
        distances = (x2 - x1).unsqueeze(0)
        frac = distances / l.unsqueeze(-1).unsqueeze(-1)
        return torch.exp(-0.5 * (frac**2))
    
    def build_covariance_matrix(self, lambda_matrix):
        l = F.softplus(self.length_scales)
        K = self.kernel(self.squared_exponential_kernel, self.n_bins, self.n_bins, l=l)
    
        blocks = torch.einsum('btok,kts,bsuk->btosu', lambda_matrix, K, lambda_matrix)
        cov = blocks.reshape(
            lambda_matrix.size(0),
            self.n_bins * self.n_units,
            self.n_bins * self.n_units
        )
    
        cov = cov + self.I.unsqueeze(0)
        noise_diag = torch.sigmoid(self.noise.flatten())
        cov.diagonal(dim1=-2, dim2=-1).add_(noise_diag)
    
        L = torch.linalg.cholesky(cov)
        return L


class IdentityCovModel(CovModel):
    def __init__(self, Conf):
        super().__init__(Conf)

    def forward(self, x):
        batch_size = x[0].size(0)
        L = self.I.unsqueeze(0).expand(batch_size, -1, -1)
        return L
            

class SharedCovModel(CovModel):
    def __init__(self, Conf):
        super().__init__(Conf)
        
        self.lambda_matrix = nn.Parameter(torch.empty(self.n_bins, self.n_units, self.n_latent, device=self.device))
        with torch.no_grad():
            nn.init.normal_(self.lambda_matrix, mean=0.0, std=1e-1)

    def forward(self, x):
        batch_size = x[0].size(0)
        L = self.build_covariance_matrix(self.lambda_matrix.unsqueeze(0)).squeeze(0)
        L = L.unsqueeze(0).expand(batch_size, -1, -1)
        return L


class FullModel(Model):
    def __init__(self, Conf, mean_model, cov_model):
        super().__init__(Conf)
        if mean_model == 'zero':
            self.mean_model = ZeroMeanModel(Conf)
        elif mean_model == 'baseline':
            self.mean_model = BaselineMeanModel(Conf)
        elif mean_model == 'conditional':
            self.mean_model = ConditionalMeanModel(Conf)

        if cov_model == 'identity':
            self.cov_model = IdentityCovModel(Conf)
        elif cov_model == 'shared':
            self.cov_model = SharedCovModel(Conf)
    
    def forward(self, x):
        mean = self.mean_model(x)
        L = self.cov_model(x)
        return mean, L


class ConditionalGaussian(Model):
    def __init__(self, Conf, variant):
        super().__init__(Conf)
        self.variant = variant

    def predict(self, mean, cov, y, **kwargs):
        if self.variant == 'mean':
            return mean
        elif self.variant == 'past':
            return self.__predict_by_past(mean, cov, y)
        elif self.variant == 'others':
            return self.__predict_by_others(mean, cov, y)
        elif self.variant == 'past_and_others':
            return self.__predict_by_past_and_others(mean, cov, y)
        else:
            raise ValueError(f'Variant is not supported: {self.variant}')
    
    def __predict_by_past(self, mean, cov, y, max_t_past_prediction=None):
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.n_units))
    
        for t in range(1, conditioned_mean.shape[0]):
            conditioned_indices = torch.arange(t * self.n_units, device=mean.device).reshape(t, self.n_units).t()
            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(t).reshape(self.n_units, t, t),
                conditioned_indices.unsqueeze(1).repeat(1, t, 1)
            ), dim=-1)
    
            row_indices = torch.arange(t * self.n_units, (t + 1) * self.n_units, device=mean.device)
            cross_cov = cov[row_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.n_units, t, t)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.n_units)
        return conditioned_mean
    
    def __predict_by_others(self, mean, cov, y):
        nr_others = self.n_units - 1
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.n_units))
    
        for t in range(conditioned_mean.shape[0]):
            time_indices = torch.arange(t * self.n_units, (t + 1) * self.n_units, device=mean.device)
            conditioned_indices = time_indices.unsqueeze(0).repeat(self.n_units, 1)
            conditioned_indices = conditioned_indices[~torch.eye(self.n_units, dtype=torch.bool, device=mean.device)].reshape(self.n_units, nr_others)
    
            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(nr_others).reshape(self.n_units, nr_others, nr_others),
                conditioned_indices.unsqueeze(1).repeat(1, nr_others, 1)
            ), dim=-1)
    
            cross_cov = cov[time_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.n_units, nr_others, nr_others)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.n_units)
        
        return conditioned_mean
    
    def __predict_by_past_and_others(self, mean, cov, y, max_t_past_prediction=None):
        nr_others = self.n_units - 1
        y_flat = y.flatten()
        mean_flat = mean.flatten()
        conditioned_mean = mean.clone().reshape((-1, self.n_units))
    
        for t in range(conditioned_mean.shape[0]):
            nr_past_and_others = t + nr_others
    
            past_indices = torch.arange(t * self.n_units, device=mean.device).reshape(t, self.n_units).t()
            time_indices = torch.arange(t * self.n_units, (t + 1) * self.n_units, device=mean.device)
            others_indices = time_indices.unsqueeze(0).repeat(self.n_units, 1)
            others_indices = others_indices[~torch.eye(self.n_units, dtype=torch.bool, device=mean.device)].reshape(self.n_units, nr_others)
    
            conditioned_indices = torch.cat((past_indices, others_indices), dim=-1)

            product_indices = torch.stack((
                conditioned_indices.flatten().repeat_interleave(nr_past_and_others).reshape(self.n_units, nr_past_and_others, nr_past_and_others),
                conditioned_indices.unsqueeze(1).repeat(1, nr_past_and_others, 1)
            ), dim=-1)
    
            cross_cov = cov[time_indices.unsqueeze(-1), conditioned_indices].unsqueeze(1)
            partial_cov = cov[product_indices[..., 0], product_indices[..., 1]].reshape(self.n_units, nr_past_and_others, nr_past_and_others)
            diff = (y_flat[conditioned_indices] - mean_flat[conditioned_indices]).unsqueeze(-1)
    
            conditioned_mean[t] += torch.matmul(torch.matmul(cross_cov, torch.linalg.inv(partial_cov)), diff).reshape(self.n_units)
        
        return conditioned_mean



def eval_mean_model(module):
    if not any(p.requires_grad for p in module.full_model.mean_model.parameters()):
        module.full_model.mean_model.eval()

class History(L.Callback):
    def __init__(self):
        self.train_loss_epoch = []
        self.valid_loss_epoch = []
        self.lr = []

    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss_epoch.append(trainer.callback_metrics["train_loss_epoch"].item())

    def on_validation_end(self, trainer, pl_module):
        self.valid_loss_epoch.append(trainer.callback_metrics["valid_loss_epoch"].item())
        self.lr.append(trainer.optimizers[0].param_groups[0]["lr"])

class LitModel(L.LightningModule):
    def __init__(self, Conf, mean_model, cov_model):
        super().__init__()
        self.save_hyperparameters(ignore=["optimizer_type", "scheduler_type", "data", "model_params"])        
        self.optimizer_params = Conf.optimization.Adam
        self.scheduler_params = Conf.optimization.Reduce
        self.full_model = FullModel(Conf, mean_model, cov_model)
        self.mvn_nll_loss = MVNNLLLoss()
        self.anscombe = Anscombe()
     
    def forward(self, x):
        return self.full_model(x)

    def training_step(self, batch):
        x, y = batch
        x = tuple(t.to(self.device, non_blocking=True) for t in x)
        y = y.to(self.device, non_blocking=True)
        
        eval_mean_model(self)
        out = self(x)
        loss = self.mvn_nll_loss(out, y).mean()
        self.log("train_loss_epoch", loss, on_step=False, on_epoch=True, logger=False, prog_bar=True)
        
        return loss

    def validation_step(self, batch):
        x, y = batch
        x = tuple(t.to(self.device, non_blocking=True) for t in x)
        y = y.to(self.device, non_blocking=True)
        
        eval_mean_model(self)
        out = self(x)
        loss = self.mvn_nll_loss(out, y).mean()
        
        self.log("valid_loss_epoch", loss, on_step=False, on_epoch=True, logger=False, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            (p for p in self.parameters() if p.requires_grad),
            lr=self.optimizer_params.lr,
            weight_decay=self.optimizer_params.weight_decay,
        )

        scheduler = ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=self.scheduler_params.factor,
            patience=self.scheduler_params.patience,
            min_lr=self.scheduler_params.min_lr,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "valid_loss_epoch",
                "interval": "epoch",
                "frequency": 1,
            },
        }

In [36]:
# @title Functions


def prepare_dataset(Conf, x_position_vars, x_dense_vars, x_sparse_vars, Y):

    seed_everything(Conf.seed)

    split_rng = np.random.default_rng(Conf.seed)
    train_idx = split_rng.choice(Conf.data.n_trials, size=int(Conf.data.n_trials * 0.8), replace=False)
    valid_idx = np.setdiff1d(np.arange(Conf.data.n_trials), train_idx)

    x_position_vars_train = x_position_vars[train_idx]
    x_position_vars_valid = x_position_vars[valid_idx]

    x_dense_vars_train = x_dense_vars[train_idx]
    x_dense_vars_valid = x_dense_vars[valid_idx]

    x_sparse_vars_train = x_sparse_vars[train_idx]
    x_sparse_vars_valid = x_sparse_vars[valid_idx]

    Y_train = Y[train_idx]
    Y_valid = Y[valid_idx]

    position_Y_mean = x_position_vars_train.mean(axis=0, keepdims=True)
    position_std = x_position_vars_train.std(axis=0, keepdims=True)
    x_position_vars_train = (x_position_vars_train - position_Y_mean) / (position_std + 1e-8)
    x_position_vars_valid = (x_position_vars_valid - position_Y_mean) / (position_std + 1e-8)

    dense_task_Y_mean = x_dense_vars_train.mean(axis=0, keepdims=True)
    dense_task_std = x_dense_vars_train.std(axis=0, keepdims=True)
    x_dense_vars_train = (x_dense_vars_train - dense_task_Y_mean) / (dense_task_std + 1e-8)
    x_dense_vars_valid = (x_dense_vars_valid - dense_task_Y_mean) / (dense_task_std + 1e-8)

    Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32, device=Conf.device).transpose(1, 2)
    anscombe = Anscombe()
    Y_train_anscombed = anscombe.forward(Y_train_tensor)
    Y_mean = torch.mean(Y_train_anscombed, dim=(0, 1), keepdim=True)    

    train_dataset = NeuralDataset(x_position_vars_train, x_dense_vars_train, x_sparse_vars_train, Y_train, Y_mean)
    valid_dataset = NeuralDataset(x_position_vars_valid, x_dense_vars_valid, x_sparse_vars_valid, Y_valid, Y_mean)

    return train_dataset, valid_dataset, Y_mean


def shuffle_dataset(Conf, dataset):

    generator = torch.Generator()
    generator.manual_seed(Conf.seed)

    indices = torch.randperm(len(dataset), generator=generator)

    dataset_shuffle = copy.deepcopy(dataset)
    dataset_shuffle.Y = dataset.Y[indices]

    return dataset_shuffle


def prepare_loader(Conf, train_dataset, valid_dataset):

    seed_everything(Conf.seed)

    train_loader_generator = torch.Generator()
    train_loader_generator.manual_seed(Conf.seed)
    train_loader = DataLoader(train_dataset, batch_size=Conf.training.batch_size, shuffle=True, generator=train_loader_generator, num_workers=0, pin_memory=True, persistent_workers=False)
    valid_loader_generator = torch.Generator()
    valid_loader_generator.manual_seed(Conf.seed)
    valid_loader = DataLoader(valid_dataset, batch_size=Conf.training.batch_size, shuffle=False, generator=valid_loader_generator, num_workers=0, pin_memory=True, persistent_workers=False)

    return train_loader, valid_loader, (train_loader_generator, valid_loader_generator)


def build_lit_model(Conf, loader_generators, mean_model, cov_model, enable_progress_bar_epoch):

    train_loader_generator, valid_loader_generator = loader_generators
    seed_everything(Conf.seed)
    train_loader_generator.manual_seed(Conf.seed)
    valid_loader_generator.manual_seed(Conf.seed)


    early_stop = EarlyStopping(
        monitor="valid_loss_epoch",
        mode="min",
        min_delta=Conf.training.min_delta,
        patience=Conf.training.patience,
    )

    history = History()
    
    trainer = L.Trainer(
        max_epochs=Conf.training.max_epoch,
        accelerator="gpu" if Conf.device.type == "cuda" else "cpu",
        devices=1,
        precision="16-mixed",
        deterministic=True,
        num_sanity_val_steps=0,
        logger=False,
        callbacks=[early_stop, history],
        enable_progress_bar=enable_progress_bar_epoch,
        enable_checkpointing=False,
        enable_model_summary=False,
        check_val_every_n_epoch=1,
    )
    
    lit_model = LitModel(Conf, mean_model, cov_model).to(Conf.device)
    trainer.history = history
    
    return trainer, lit_model


def compute_shap_values(Conf, lit_model, background_dataset, explain_dataset, n_permutations=100):

    seed = Conf.seed
    seed_everything(seed)
    device = Conf.device
    shap_generator = torch.Generator(device=device)
    shap_generator.manual_seed(Conf.seed)

    n_bins = Conf.data.n_bins
    n_units = Conf.data.n_units
    n_vars = Conf.data.n_vars
    n_states = n_vars + 1
    n_background_trials = len(background_dataset)
    n_explain_trials = len(explain_dataset)

    mean_model = lit_model.full_model.mean_model.to(device).eval()

    background = [background_dataset.x_position_vars.to(device), background_dataset.x_dense_vars.to(device), background_dataset.x_sparse_vars.to(device)]
    explain = [explain_dataset.x_position_vars.to(device), explain_dataset.x_dense_vars.to(device), explain_dataset.x_sparse_vars.to(device)]

    shap_values = torch.empty(n_explain_trials, n_vars, n_bins, n_units, device=device)
    base_values = torch.empty(n_explain_trials, n_bins, n_units, device=device)

    with torch.inference_mode():

        for explain_trial_idx in range(n_explain_trials):

            permutation = torch.rand(n_permutations, n_vars, device=device, generator=shap_generator).argsort(dim=1)
            order = permutation.argsort(dim=1)
            
            included = order[:, None] < torch.arange(n_states, device=device)[None, :, None]

            background_idx = torch.randint(n_background_trials, (n_permutations,), device=device, generator=shap_generator)

            position = torch.where(included[:, :, None, :4], explain[0][[explain_trial_idx], None, :, :].clone(), background[0][background_idx, None, :, :].clone())
            dense = torch.where(included[:, :, 4:8], explain[1][[explain_trial_idx], None, :].clone(), background[1][background_idx, None, :].clone())
            sparse = torch.where(included[:, :, 8:], explain[2][[explain_trial_idx], None, :].clone(), background[2][background_idx, None, :].clone())

            outputs = mean_model((position.flatten(0, 1), dense.flatten(0, 1), sparse.flatten(0, 1))).reshape(n_permutations, n_states, n_bins, n_units)

            contributions = outputs[:, 1:] - outputs[:, :-1]

            values = torch.zeros(n_vars, n_bins, n_units, device=device)
            values.index_add_(0, permutation.reshape(-1), contributions.flatten(0, 1))

            shap_values[explain_trial_idx] = values / n_permutations
            base_values[explain_trial_idx] = outputs[:, 0].mean(dim=0)

    return [i.cpu().numpy() for i in background], [i.cpu().numpy() for i in explain], shap_values.cpu().numpy(), base_values.cpu().numpy()


def predict_loader(Conf, lit_model, loader, Y_mean, variant):
    device = Conf.device
    lit_model = lit_model.to(device)
    lit_model.eval()

    conditional_gaussian = ConditionalGaussian(Conf, variant).to(device)
    anscombe = Anscombe().to(device)
    Y_mean = torch.as_tensor(Y_mean, dtype=torch.float32, device=device).mean(dim=0, keepdim=True)

    Y = []
    Y_hat = []

    with torch.inference_mode():
        for x, y in loader:
            x = tuple(t.to(device, non_blocking=True) for t in x)
            y = y.to(device, non_blocking=True)

            mean, L = lit_model(x)
            cov = L @ L.transpose(-1, -2)

            y_hat = torch.stack([
                conditional_gaussian.predict(mean_sample, cov_sample, y_sample)
                for mean_sample, cov_sample, y_sample in zip(mean, cov, y)
            ])

            Y.append(anscombe.inv(y + Y_mean))
            Y_hat.append(anscombe.inv(y_hat + Y_mean))

    Y = torch.cat(Y).cpu().numpy()
    Y_hat = torch.cat(Y_hat).cpu().numpy()

    return Y, Y_hat


compute_correlation = lambda y, y_hat: np.corrcoef(y.flatten(), y_hat.flatten())[0, 1] if np.std(y)>0 and np.std(y_hat)>0 else 0.0
compute_r2 = lambda y, y_hat: 1 - np.sum((y.flatten() - y_hat.flatten()) ** 2) / (np.sum((y.flatten() - np.mean(y.flatten())) ** 2) + 1e-8)
compute_mse = lambda y, y_hat: np.mean((y - y_hat) ** 2)


def compute_metrics(Conf, Y, Y_hat):
    n_units = Conf.data.n_units
    
    correlations_trial = []
    r2s_trial = []
    mses_trial = []

    for unit_idx in range(n_units):
        y = Y[:, :, unit_idx]
        y_hat = Y_hat[:, :, unit_idx]
        
        corr = [compute_correlation(y_trial, y_hat_trial) for y_trial, y_hat_trial in zip(y, y_hat)]
        r2 = [compute_r2(y_trial, y_hat_trial) for y_trial, y_hat_trial in zip(y, y_hat)]
        mse = [compute_mse(y_trial, y_hat_trial) for y_trial, y_hat_trial in zip(y, y_hat)]
        
        correlations_trial.append(corr)
        r2s_trial.append(r2)
        mses_trial.append(mse)

    return correlations_trial, r2s_trial, mses_trial


In [ ]:
# @title Plots

def save_figure(fig, file_name, file_path, ext=".png", **savefig_kwargs):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    fig.savefig(file_path / f"{file_name}{ext}", dpi=100, bbox_inches="tight", pad_inches=0.25, **savefig_kwargs)

    plt.close(fig)

    return file_path


def save_shap_plot(make_plot, title, file_name, file_path, figsize=(10, 6)):
    plt.close("all")

    plot_result = make_plot()

    if hasattr(plot_result, "figure"):
        fig = plot_result.figure
        ax = plot_result
    else:
        fig = plt.gcf()
        ax = plt.gca()

    ax.set_title(title, pad=20, fontweight="bold")
    fig.canvas.draw()

    return save_figure(fig, file_name, file_path)


def plot_training_history(
    trainer,
    title,
    file_name,
    file_path,
    show=False,
):

    h = trainer.history
    train = np.asarray(h.train_loss_epoch)
    valid = np.asarray(h.valid_loss_epoch)
    lr = np.asarray(h.lr)
    epochs = np.arange(1, len(valid) + 1)

    trend = lambda x: np.diff(x) / np.maximum(np.abs(x[:-1]), 1e-8)

    fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")

    ax[0].plot(epochs, valid, color="tab:orange", label="Validation")
    ax[0].plot(epochs, train, color="tab:blue", label="Train")
    ax[0].set(title="Loss", xlabel="Epoch", ylabel="NLL loss")
    ax[0].legend(frameon=False)

    ax[1].axhline(0, color="black", ls="--", lw=1)
    ax[1].plot(epochs[1:], trend(valid), color="tab:orange", label="Validation")
    ax[1].plot(epochs[1:], trend(train), color="tab:blue", label="Train")
    ax[1].set(title="Loss trend", xlabel="Epoch", ylabel="Relative change")
    ax[1].legend(frameon=False)

    ax[2].plot(epochs, lr, color="tab:green")
    ax[2].set(title="Learning rate", xlabel="Epoch", ylabel="LR")
    ax[2].set_yscale("log")

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_cov_loading_matrix(
    mean_cov_lit_model,
    title,
    file_name,
    file_path,
    bin_times=None,
    vmin=None,
    vmax=None,
    show=False,
):

    lambda_matrix = mean_cov_lit_model.full_model.cov_model.lambda_matrix
    lambda_matrix = lambda_matrix.detach().cpu().numpy()

    n_bins, n_units, n_latent = lambda_matrix.shape

    if bin_times is None:
        bin_times = np.arange(n_bins)

    loading_matrix = lambda_matrix.transpose(2, 0, 1).reshape(n_latent, -1)

    if vmin is None and vmax is None:
        vmax = np.max(np.abs(loading_matrix))
        vmin = -vmax
    elif vmin is None:
        vmin = -vmax
    elif vmax is None:
        vmax = -vmin

    fig, ax = plt.subplots(
        figsize=(max(12, n_bins * 1.2), max(4, n_latent * 0.35)),
        layout="constrained",
    )

    im = ax.imshow(
        loading_matrix,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
    )

    for bin_idx in range(1, n_bins):
        ax.axvline(
            bin_idx * n_units - 0.5,
            color="black",
            linewidth=0.7,
            alpha=0.7,
        )

    bin_centers = np.arange(n_bins) * n_units + (n_units - 1) / 2

    ax.set(
        xlabel="Time bin",
        ylabel="Latent",
        xticks=bin_centers,
        xticklabels=np.round(bin_times, 2),
        yticks=np.arange(n_latent),
        yticklabels=np.arange(n_latent),
    )

    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=8)

    cbar = fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
    cbar.set_label(r"$\lambda$", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_cov_noise(
    mean_cov_lit_model,
    title,
    file_name,
    file_path,
    bin_times=None,
    unit_names=None,
    vmax=None,
    show=False,
):

    noise = mean_cov_lit_model.full_model.cov_model.noise
    noise = torch.sigmoid(noise).detach().cpu().numpy()

    n_bins, n_units = noise.shape

    if bin_times is None:
        bin_times = np.arange(n_bins)
    if unit_names is None:
        unit_names = np.arange(n_units)
    if vmax is None:
        vmax = np.max(noise)

    fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")

    im = ax.imshow(
        noise.T,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap="magma",
        vmin=0,
        vmax=vmax,
    )

    ax.set(
        xlabel="Time",
        ylabel="Unit",
        xticks=np.arange(n_bins),
        xticklabels=np.round(bin_times, 2),
        yticks=np.arange(n_units),
        yticklabels=unit_names,
    )

    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Noise variance", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_cov_length_scales(
    mean_cov_lit_model,
    title,
    file_name,
    file_path,
    ymax=None,
    show=False,
):

    length_scales = mean_cov_lit_model.full_model.cov_model.length_scales
    length_scales = F.softplus(length_scales).detach().cpu().numpy()

    if ymax is None:
        ymax = np.max(length_scales)

    fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")

    ax.bar(np.arange(len(length_scales)), length_scales, color="C0")

    ax.set(
        xlabel="Latent",
        ylabel="Length scale",
        xticks=np.arange(len(length_scales)),
        ylim=(0, ymax),
    )

    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_covariance_matrix(
    mean_cov_lit_model,
    title,
    file_name,
    file_path,
    bin_times=None,
    time_indices=None,
    vmin=None,
    vmax=None,
    linthresh=None,
    show=False,
):
    lambda_matrix = mean_cov_lit_model.full_model.cov_model.lambda_matrix.unsqueeze(0)
    cov_tensor = mean_cov_lit_model.full_model.cov_model.build_covariance_matrix(lambda_matrix)
    cov_matrix = (cov_tensor @ cov_tensor.transpose(-1, -2)).squeeze(0).detach().cpu().numpy()
    
    n_bins, n_units = mean_cov_lit_model.full_model.cov_model.n_bins, mean_cov_lit_model.full_model.cov_model.n_units
    
    if bin_times is None:
        bin_times = np.arange(n_bins)

    if time_indices is None:
        time_indices = np.arange(n_bins)
    else:
        time_indices = np.atleast_1d(time_indices)

    feature_indices = np.concatenate([
        np.arange(time_idx * n_units, (time_idx + 1) * n_units)
        for time_idx in time_indices
    ])

    cov_matrix = cov_matrix[np.ix_(feature_indices, feature_indices)]
    bin_times = np.asarray(bin_times)[time_indices]
    n_bins = len(time_indices)
        
    if vmin is None and vmax is None:
        vmax = np.max(np.abs(cov_matrix))
        vmin = -vmax
    elif vmin is None:
        vmin = -vmax
    elif vmax is None:
        vmax = -vmin

    if linthresh is None:
        linthresh = max(vmax * 0.01, 1e-8)

    norm = mcolors.SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)

    fig, ax = plt.subplots(figsize=(max(10, n_bins * 0.5), max(10, n_bins * 0.5)), layout="constrained")

    im = ax.imshow(
        cov_matrix,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
        cmap="RdBu_r",
        norm=norm,
    )

    for bin_idx in range(1, n_bins):
        line_pos = bin_idx * n_units - 0.5
        ax.axvline(line_pos, color="black", linewidth=0.5, alpha=0.5)
        ax.axhline(line_pos, color="black", linewidth=0.5, alpha=0.5)

    bin_centers = np.arange(n_bins) * n_units + (n_units - 1) / 2
    tick_labels = np.round(bin_times, 2)

    ax.set(
        xticks=bin_centers,
        xticklabels=tick_labels,
        yticks=bin_centers,
        yticklabels=tick_labels,
    )
    
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Covariance (SymLog Scale)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_prediction(
    Y,
    Y_hat,
    trial_idx,
    unit_idx,
    bin_times,
    title,
    filename,
    filepath,
    figsize=(10, 2),
    show=False,
):
    to_numpy = lambda x: (
        x.detach().cpu().numpy()
        if torch.is_tensor(x)
        else np.asarray(x)
    )

    y = to_numpy(Y[trial_idx])[:, unit_idx]
    y_hat = to_numpy(Y_hat[trial_idx])[:, unit_idx]
    bin_times = np.asarray(bin_times)

    fig, ax = plt.subplots(figsize=figsize, layout="constrained")

    ax.plot(
        bin_times,
        y,
        color="tab:green",
        linewidth=2,
        label="Ground truth",
    )
    ax.plot(
        bin_times,
        y_hat,
        color="tab:purple",
        linewidth=2,
        label="GRU prediction",
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set(
        xlabel="Time from press onset (ms)",
        ylabel="Neural activity",
    )
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, filepath)


def plot_train_valid_metrics_comparison(
    correlations_train,
    correlations_valid,
    r2s_train,
    r2s_valid,
    mses_train,
    mses_valid,
    title,
    filename,
    filepath,
    bins=20,
    figsize=(14, 7),
    show=False,
):
    metrics = [
        (
            "Correlation",
            np.asarray(correlations_train, dtype=float),
            np.asarray(correlations_valid, dtype=float),
            "tab:brown",
        ),
        (
            r"$R^2$",
            np.asarray(r2s_train, dtype=float),
            np.asarray(r2s_valid, dtype=float),
            "tab:brown",
        ),
        (
            "MSE",
            np.asarray(mses_train, dtype=float),
            np.asarray(mses_valid, dtype=float),
            "tab:brown",
        ),
    ]

    fig, axes = plt.subplots(2, 3, figsize=figsize, layout="constrained")

    for idx, (metric_name, train, valid, color) in enumerate(metrics):
        finite = np.isfinite(train) & np.isfinite(valid)
        train = train[finite]
        valid = valid[finite]

        lower = min(train.min(), valid.min())
        upper = max(train.max(), valid.max())
        padding = max(0.05 * (upper - lower), 1e-8)
        limits = (lower - padding, upper + padding)

        ax = axes[0, idx]

        ax.scatter(
            train,
            valid,
            s=32,
            alpha=0.7,
            color=color,
            edgecolor="none",
        )
        ax.plot(limits, limits, color="black", linestyle="--", linewidth=1)
        ax.set(
            title=metric_name,
            xlabel=f"Train {metric_name}",
            ylabel=f"Validation {metric_name}",
            xlim=limits,
            ylim=limits,
            aspect="equal",
        )
        ax.spines[["top", "right"]].set_visible(False)

        difference = train - valid
        limit = max(np.max(np.abs(difference)), 1e-8)
        edges = np.linspace(-limit, limit, bins + 1)

        ax = axes[1, idx]

        ax.hist(
            difference,
            bins=edges,
            color=color,
            alpha=0.7,
            edgecolor="white",
        )
        ax.axvline(0, color="gray", linestyle="--", linewidth=1)
        ax.set(
            xlabel=f"Train - validation {metric_name}",
            ylabel="Number of units",
            xlim=(-limit, limit),
        )
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, filepath)


def plot_model_metrics_comparison(
    correlations_x_train,
    correlations_x_valid,
    r2s_x_train,
    r2s_x_valid,
    mses_x_train,
    mses_x_valid,
    correlations_y_train,
    correlations_y_valid,
    r2s_y_train,
    r2s_y_valid,
    mses_y_train,
    mses_y_valid,
    x_label,
    y_label,
    title,
    file_name,
    file_path,
    bins=20,
    figsize=(14, 7),
    show=False,
):
    metrics = [
        (
            "Correlation",
            correlations_x_train,
            correlations_x_valid,
            correlations_y_train,
            correlations_y_valid,
        ),
        (
            r"$R^2$",
            r2s_x_train,
            r2s_x_valid,
            r2s_y_train,
            r2s_y_valid,
        ),
        (
            "MSE",
            mses_x_train,
            mses_x_valid,
            mses_y_train,
            mses_y_valid,
        ),
    ]

    fig, axes = plt.subplots(2, 3, figsize=figsize, layout="constrained")

    for idx, (metric_name, x_train, x_valid, y_train, y_valid) in enumerate(metrics):
        x_train = np.asarray(x_train, dtype=float)
        x_valid = np.asarray(x_valid, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        y_valid = np.asarray(y_valid, dtype=float)

        finite_train = np.isfinite(x_train) & np.isfinite(y_train)
        finite_valid = np.isfinite(x_valid) & np.isfinite(y_valid)

        x_train, y_train = x_train[finite_train], y_train[finite_train]
        x_valid, y_valid = x_valid[finite_valid], y_valid[finite_valid]

        lower = min(x_train.min(), x_valid.min(), y_train.min(), y_valid.min())
        upper = max(x_train.max(), x_valid.max(), y_train.max(), y_valid.max())
        padding = max(0.05 * (upper - lower), 1e-8)
        limits = (lower - padding, upper + padding)

        ax = axes[0, idx]

        ax.scatter(
            x_train,
            y_train,
            s=32,
            alpha=0.7,
            color="tab:blue",
            edgecolor="none",
            label="Training",
        )
        ax.scatter(
            x_valid,
            y_valid,
            s=32,
            alpha=0.7,
            color="tab:orange",
            edgecolor="none",
            label="Validation",
        )
        ax.plot(limits, limits, color="gray", linestyle="--", linewidth=1.5)
        ax.set(
            title=metric_name,
            xlabel=f"{x_label} {metric_name}",
            ylabel=f"{y_label} {metric_name}",
            xlim=limits,
            ylim=limits,
            aspect="equal",
        )
        ax.legend(frameon=False)
        ax.spines[["top", "right"]].set_visible(False)

        difference_train = x_train - y_train
        difference_valid = x_valid - y_valid
        limit = max(
            np.max(np.abs(difference_train)),
            np.max(np.abs(difference_valid)),
            1e-8,
        )
        edges = np.linspace(-limit, limit, bins + 1)

        ax = axes[1, idx]

        ax.hist(
            difference_train,
            bins=edges,
            color="tab:blue",
            alpha=0.6,
            edgecolor="white",
            label="Training",
        )
        ax.hist(
            difference_valid,
            bins=edges,
            color="tab:orange",
            alpha=0.6,
            edgecolor="white",
            label="Validation",
        )
        ax.axvline(0, color="gray", linestyle="--", linewidth=1.5)
        ax.set(
            xlabel=f"{x_label} - {y_label} {metric_name}",
            ylabel="Number of units",
            xlim=(-limit, limit),
        )
        ax.legend(frameon=False)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_model_metric_improvement(
        mse_trial_baseline, 
        mse_trial_model,
        corr_trial_baseline, 
        corr_trial_model,
        r2_trial_baseline, 
        r2_trial_model,
        title, 
        file_name, 
        file_path, 
        alpha=0.05, 
        show=False
):

    
    n_units = len(mse_trial_baseline)
    
    p_values_mse, p_values_corr, p_values_r2 = [], [], []
    better_mse, better_corr, better_r2 = [], [], []

    for unit_idx in range(n_units):
        mse_b = mse_trial_baseline[unit_idx]
        mse_m = mse_trial_model[unit_idx]
        corr_b = corr_trial_baseline[unit_idx]
        corr_m = corr_trial_model[unit_idx]
        r2_b = r2_trial_baseline[unit_idx]
        r2_m = r2_trial_model[unit_idx]
        
        try:
            _, p_corr = wilcoxon(corr_b, corr_m)
        except ValueError: 
            p_corr = 1.0 
            
        try:
            _, p_r2 = wilcoxon(r2_b, r2_m)
        except ValueError:
            p_r2 = 1.0
            
        try:
            _, p_mse = wilcoxon(mse_b, mse_m)
        except ValueError:
            p_mse = 1.0
            
        p_values_corr.append(p_corr)
        p_values_r2.append(p_r2)
        p_values_mse.append(p_mse)
        
        better_corr.append(np.nanmean(corr_m) > np.nanmean(corr_b))
        better_r2.append(np.nanmean(r2_m) > np.nanmean(r2_b))
        better_mse.append(np.nanmean(mse_m) < np.nanmean(mse_b))

    def get_proportions(p_vals, is_better):
        sig = sum(1 for p, better in zip(p_vals, is_better) if p < alpha and better)
        total = len(p_vals)
        return sig / total, (total - sig) / total

    sig_mse, nonsig_mse = get_proportions(p_values_mse, better_mse)
    sig_corr, nonsig_corr = get_proportions(p_values_corr, better_corr)
    sig_r2, nonsig_r2 = get_proportions(p_values_r2, better_r2)
    
    metrics = ['Loss', 'Correlation', 'R²']
    significant = [sig_mse, sig_corr, sig_r2]
    non_significant = [nonsig_mse, nonsig_corr, nonsig_r2]
    
    fig, ax = plt.subplots(figsize=(6, 6))
    
    p1 = ax.bar(metrics, significant, color='#4558C4', label='Significant')
    p2 = ax.bar(metrics, non_significant, bottom=significant, color='#C42A2F', label='Non-significant')
    
    ax.bar_label(p1, label_type='center', fmt='%.2f', fontsize=12)
    ax.bar_label(p2, label_type='center', fmt='%.2f', fontsize=12)
    
    ax.set_ylim(0, 1)
    ax.set_ylabel('Percentage', fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.7, color='grey', linewidth=1)
    ax.set_axisbelow(True)
    
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncol=2, frameon=False, fontsize=14)    
    fig.suptitle(title, fontweight='bold')
    
    if show:
        plt.show()
        
    return save_figure(fig, file_name, file_path)


def plot_shap(
    shap_values,
    title,
    file_name, 
    file_path,
    bin_times=None,
    unit_names=None,
    cmap="RdBu_r",
    figsize=(10, 7),
    show=False,
):
    shap_values = np.asarray(shap_values, dtype=float)

    if shap_values.ndim != 2:
        raise ValueError(
            "shap_values must have shape (n_bins, n_units). "
            f"Received shape {shap_values.shape}."
        )

    n_bins, n_units = shap_values.shape

    if bin_times is None:
        bin_times = np.arange(n_bins)
    if unit_names is None:
        unit_names = np.arange(n_units)

    bin_times = np.asarray(bin_times)
    unit_names = np.asarray(unit_names)

    if bin_times.size != n_bins:
        raise ValueError("bin_times must contain one value per bin.")
    if unit_names.size != n_units:
        raise ValueError("unit_names must contain one name per unit.")

    if cmap == "RdBu_r":
        vmin = -np.nanmax(np.abs(shap_values))
        vmax = np.nanmax(np.abs(shap_values))
    else:
        vmin = 0
        vmax = np.nanmax(shap_values)

    mean_by_unit = np.nanmean(shap_values, axis=0)
    mean_by_bin = np.nanmean(shap_values, axis=1)

    fig = plt.figure(figsize=figsize, layout="constrained")
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=[0.25, 1],
        height_ratios=[1, 0.25],
        wspace=0.03,
        hspace=0.03,
    )

    ax_main = fig.add_subplot(gs[0, 1])
    ax_left = fig.add_subplot(gs[0, 0], sharey=ax_main)
    ax_bottom = fig.add_subplot(gs[1, 1])
    ax_corner = fig.add_subplot(gs[1, 0])
    ax_corner.axis("off")

    im = ax_main.imshow(
        shap_values.T,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )

    ax_main.set_title(title, fontweight="bold")
    ax_main.set_ylabel("Unit")
    ax_main.set_xticks([])
    ax_main.set_yticks(np.arange(n_units))
    ax_main.set_yticklabels(unit_names, fontsize=7)
    ax_main.spines[["top", "right"]].set_visible(False)

    unit_idx = np.arange(n_units)

    ax_left.axvline(0, color="black", linewidth=0.8, zorder=0)

    ax_left.hlines(
        y=unit_idx,
        xmin=0,
        xmax=mean_by_unit,
        color="C0",
        linewidth=1.1,
        zorder=1,
    )

    ax_left.scatter(
        mean_by_unit,
        unit_idx,
        color="C0",
        s=14,
        zorder=2,
    )

    left_limit = max(np.nanmax(np.abs(mean_by_unit)), np.finfo(float).eps)
    ax_left.set_xlim(-1.1 * left_limit, 1.1 * left_limit)

    ax_left.set_xlabel("Mean SHAP", fontsize=8)
    ax_left.tick_params(axis="x", labelsize=7)
    ax_left.tick_params(axis="y", left=False, labelleft=False)
    ax_left.spines[["top", "right", "left"]].set_visible(False)

    ax_bottom.axhline(0, color="black", linewidth=0.8, zorder=0)

    ax_bottom.plot(
        bin_times,
        mean_by_bin,
        color="C0",
        linewidth=1.5,
    )

    ax_bottom.fill_between(
        bin_times,
        0,
        mean_by_bin,
        color="C0",
        alpha=0.20,
    )

    ax_bottom.set_xlabel("Time")
    ax_bottom.set_ylabel("Mean\nSHAP", fontsize=8)
    ax_bottom.tick_params(axis="both", labelsize=7)
    ax_bottom.spines[["top", "right"]].set_visible(False)

    cbar = fig.colorbar(
        im,
        ax=ax_main,
        fraction=0.025,
        pad=0.02,
    )
    cbar.set_label("SHAP value", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    if show:
        plt.show()

    return save_figure(fig, file_name, file_path)


def plot_unit_selectivity_hist(
    unit_selectivity_shuffles,
    unit_selectivity,
    class_names,
    unit_name,
    title,
    filename,
    file_path="./plot/selectivity/units",
    bins=30,
    show=False
):
    colors = ["gray", "green", "red", "blue"]

    fig, axes = plt.subplots(
        1,
        len(class_names),
        figsize=(4 * len(class_names), 4),
        layout="constrained"
    )

    axes = np.atleast_1d(axes)

    for ax, color, class_name, null_values, real_value in zip(
        axes,
        colors,
        class_names,
        unit_selectivity_shuffles.T,
        unit_selectivity
    ):
        null_values = null_values[np.isfinite(null_values)]

        ax.hist(
            null_values,
            bins=bins,
            density=True,
            color=color,
            edgecolor=color,
            alpha=0.25
        )

        ax.axvline(
            real_value,
            color=color,
            linewidth=2.5,
            label="Real"
        )

        ax.set_title(class_name.replace("_", " ").title())
        ax.set_xlabel("Mean absolute SHAP")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Density")
    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, file_path)


def plot_unit_class_selectivity_matrix(
    unit_selectivities,
    class_names,
    unit_names,
    selectivity_significant,
    title,
    filename,
    file_path,
    show=False
):
    base_colors = ["gray", "green", "red", "blue"]

    unit_scores = np.nanmean(unit_selectivities, axis=0)
    sort_idx = np.argsort(-unit_scores)

    sorted_matrix = unit_selectivities[:, sort_idx]
    sorted_significant = selectivity_significant[:, sort_idx]
    sorted_unit_names = unit_names[sort_idx]

    vmin = np.nanmin(sorted_matrix)
    vmax = np.nanmax(sorted_matrix)

    n_classes, n_units = sorted_matrix.shape

    fig, ax = plt.subplots(
        1,
        1,
        figsize=(max(8, 0.15 * n_units), 4),
        layout="constrained"
    )

    row_height = 0.7
    row_gap = 0.3

    for row_idx, color in enumerate(base_colors[:n_classes]):
        row_values = sorted_matrix[row_idx:row_idx + 1]

        cmap = mcolors.LinearSegmentedColormap.from_list(
            f"class_{row_idx}",
            ["white", color],
            N=256
        )

        center = row_idx * (row_height + row_gap)
        ax.imshow(
            row_values,
            aspect="auto",
            interpolation="nearest",
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            extent=(
                -0.5,
                n_units - 0.5,
                center - row_height / 2.0,
                center + row_height / 2.0
            )
        )

        selected_units = np.where(sorted_significant[row_idx])[0]

        for unit_idx in selected_units:
            ax.add_patch(
                plt.Rectangle(
                    (unit_idx - 0.5, center - row_height / 2.0),
                    1.0,
                    row_height,
                    fill=False,
                    edgecolor="black",
                    linewidth=0.8
                )
            )

    row_positions = [
        i * (row_height + row_gap) for i in range(n_classes)
    ]

    ax.set_xticks(np.arange(n_units))
    ax.set_xticklabels(sorted_unit_names, rotation=90, fontsize=7)
    ax.set_yticks(row_positions)
    ax.set_yticklabels(class_names, fontsize=9)

    ax.set_xlim(-0.5, n_units - 0.5)
    ax.set_ylim(
        -row_gap,
        (n_classes - 1) * (row_height + row_gap) + row_height + row_gap
    )

    ax.set_xlabel("Units")
    ax.set_ylabel("Classes")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, file_path)


def plot_unit_selectivity_curves(
    unit_selectivities,
    class_names,
    selectivity_significant,
    title,
    filename,
    file_path,
    show=False
):
    colors = ["gray", "green", "red", "blue"]

    fig, axes = plt.subplots(
        1,
        unit_selectivities.shape[0],
        figsize=(4 * unit_selectivities.shape[0], 4),
        layout="constrained"
    )

    axes = np.atleast_1d(axes)

    for ax, color, unit_selectivity, class_name, significant in zip(
        axes,
        colors,
        unit_selectivities,
        class_names,
        selectivity_significant
    ):
        sort_idx = np.argsort(unit_selectivity)
        sorted_unit_selectivity = unit_selectivity[sort_idx]
        sorted_significant = significant[sort_idx]

        x = np.arange(1, len(sorted_unit_selectivity) + 1)
        split_idx = np.argmax(sorted_significant) if sorted_significant.any() else len(sorted_significant) - 1

        ax.plot(
            x[:split_idx + 1],
            sorted_unit_selectivity[:split_idx + 1],
            color=color,
            linewidth=2.5,
            alpha=0.3
        )

        ax.plot(
            x[split_idx:],
            sorted_unit_selectivity[split_idx:],
            color=color,
            linewidth=2.5,
            alpha=1.0
        )

        ax.set_title(class_name.replace("_", " ").title())
        ax.set_xlabel("Units")
        ax.set_xlim(1, len(sorted_unit_selectivity))
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("effect size")
    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, file_path)


def plot_unit_selectivity_pca(
    unit_selectivities,
    class_names,
    selectivity_significant,
    title,
    filename,
    file_path,
    show=False
):
    colors = ["gray", "green", "red", "blue"]

    unit_features = unit_selectivities.T
    unit_coordinates = PCA(n_components=2).fit_transform(unit_features)

    fig, axes = plt.subplots(
        1,
        unit_selectivities.shape[0],
        figsize=(4 * unit_selectivities.shape[0], 4),
        sharex=True,
        sharey=True,
        layout="constrained"
    )

    axes = np.atleast_1d(axes)

    for ax, color, class_name, unit_selectivity, significant in zip(
        axes,
        colors,
        class_names,
        unit_selectivities,
        selectivity_significant
    ):
        cmap = mcolors.LinearSegmentedColormap.from_list(
            class_name,
            ["white", color],
            N=256
        )

        edgecolors = ["black" if sig else "gray" for sig in significant]

        ax.scatter(
            unit_coordinates[:, 0],
            unit_coordinates[:, 1],
            c=unit_selectivity,
            cmap=cmap,
            vmin=np.nanmin(unit_selectivity),
            vmax=np.nanmax(unit_selectivity),
            s=70,
            edgecolors=edgecolors,
            linewidths=1.2
        )

        ax.set_title(class_name.replace("_", " ").title())
        ax.set_xlabel("PC1")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("PC2")
    fig.suptitle(title, fontweight="bold")

    if show:
        plt.show()

    return save_figure(fig, filename, file_path)



In [10]:
data = sio.loadmat(Conf.paths.data, squeeze_me=True, struct_as_record=False)

session_name = data['session_names'][Conf.session_idx]
spikes = data['spikes'][Conf.session_idx]
position_vars = data['position_vars'][Conf.session_idx]
position_var_names = data['position_var_names']
task_vars = data['task_vars'][Conf.session_idx]
task_var_names = data['task_var_names']
events = data['events'][Conf.session_idx]
event_names = data['event_names']
unit_names = data['unit_names'][Conf.session_idx]
channel_names = data['channel_names'][Conf.session_idx]
unit_types = data['unit_types'][Conf.session_idx]
bin_size = data['bin_size']
bin_times = data['bin_times']
variable_names = np.concatenate((position_var_names, task_var_names), axis=0)

spikes  = np.transpose(spikes, (1, 0, 2)) * 0.2
position_vars  = np.transpose(position_vars, (1, 0, 2))
task_vars = np.transpose(task_vars, (1, 0))

dense = [0, 1, 2, 3]
sparse = [4, 5, 6]

In [11]:
units_to_remove = [23, 52, 61, 105, 106, 116, 118, 133, 139, 149, 150, 165, 166]
units_mask_delete = np.isin(unit_names, units_to_remove)
unit_names = unit_names[~units_mask_delete]
spikes = spikes[:, ~units_mask_delete]

In [12]:
spike_mask_finite = np.isfinite(spikes).all(axis=(1, 2))
position_vars_mask_finite = np.isfinite(position_vars).all(axis=(1, 2))
task_vars_mask_finite = np.isfinite(task_vars).all(axis=1)
mask_finite = spike_mask_finite & position_vars_mask_finite & task_vars_mask_finite


position_vars_mask_zero = (position_vars == 0).all(axis=(1, 2))

task_vars_mask_over = np.zeros(task_vars.shape[0], dtype=bool)
for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]
    task_var_mask_over = task_vars[:, task_var_idx] > 100
    task_vars_mask_over |= task_var_mask_over

trials_mask_keep = mask_finite & ~position_vars_mask_zero & ~task_vars_mask_over

spikes = spikes[trials_mask_keep]
position_vars = position_vars[trials_mask_keep]
task_vars = task_vars[trials_mask_keep]

for task_var_name in ['tunp', 'tslp']:
    task_var_idx = np.where(task_var_names == task_var_name)[0][0]
    task_vars[:, task_var_idx] = np.log(task_vars[:, task_var_idx])

scaler = MinMaxScaler(feature_range=(0, 1))
task_vars = scaler.fit_transform(task_vars)

n_trials, n_units , n_bins = spikes.shape
n_trials, n_position_vars, n_bins = position_vars.shape
n_trials, n_task_vars = task_vars.shape

Conf.data = DotDict()
Conf.data.bin_size = bin_size
Conf.data.n_bins = n_bins
Conf.data.n_position_vars = n_position_vars
Conf.data.n_dense_vars = len(dense)
Conf.data.n_sparse_vars = len(sparse)
Conf.data.n_vars = n_position_vars + len(dense) + len(sparse)
Conf.data.n_units = n_units
Conf.data.n_trials = n_trials

In [16]:
x_dense_vars = task_vars[:, dense]
x_sparse_vars = task_vars[:, sparse]
x_position_vars = position_vars
Y = spikes

train_dataset, valid_dataset, Y_mean = prepare_dataset(Conf, x_position_vars, x_dense_vars, x_sparse_vars, Y)
train_loader, valid_loader, loader_generators = prepare_loader(Conf, train_dataset, valid_dataset)

In [17]:
baseline_trainer, baseline_lit_model = build_lit_model(
    Conf, 
    loader_generators, 
    "baseline", 
    "identity", 
    True
)

baseline_trainer.fit(baseline_lit_model, train_loader, valid_loader)

plot_training_history(
    trainer=baseline_trainer,
    title="Baseline Model: Training History",
    file_name="training-history",
    file_path="./plot/model/baseline/",
)

Output()

WindowsPath('plot/model/baseline')

In [18]:
mean_trainer, mean_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "identity",
    True,
)

mean_trainer.fit(mean_lit_model, train_loader, valid_loader)

plot_training_history(
    trainer=mean_trainer,
    title="Mean Model: Training History",
    file_name="training-history",
    file_path="./plot/model/mean/",
)

Output()

WindowsPath('plot/model/mean')

In [19]:
mean_cov_trainer, mean_cov_lit_model = build_lit_model(
    Conf,
    loader_generators,
    "conditional",
    "shared",
    True,
)

mean_lit_model_state_dict = mean_lit_model.full_model.mean_model.state_dict()

mean_cov_lit_model.full_model.mean_model.load_state_dict(
    mean_lit_model_state_dict,
    strict=True,
)

for p in mean_cov_lit_model.full_model.mean_model.parameters():
    p.requires_grad = False

mean_cov_lit_model_pre = copy.deepcopy(mean_cov_lit_model)

mean_cov_trainer.fit(mean_cov_lit_model, train_loader, valid_loader)

plot_training_history(
    trainer=mean_cov_trainer,
    title="Mean-Cov Model: Training History",
    file_name="training-history",
    file_path="./plot/model/mean-cov/",
)

Output()

WindowsPath('plot/model/mean-cov')

In [21]:
plot_cov_loading_matrix(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Mean-Cov Model Before Training: Latent Covariance Loadings",
    file_name="latent-covariance-loadings-pre",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    vmin=-mean_cov_lit_model.full_model.cov_model.lambda_matrix.detach().abs().max().item(),
    vmax=mean_cov_lit_model.full_model.cov_model.lambda_matrix.detach().abs().max().item(),
)

plot_cov_noise(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Mean-Cov Model Before Training: Noise Variance",
    file_name="noise-variance-pre",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    unit_names=unit_names,
    vmax=torch.sigmoid(mean_cov_lit_model.full_model.cov_model.noise).detach().max().item(),
)

plot_cov_length_scales(
    mean_cov_lit_model=mean_cov_lit_model_pre,
    title="Mean-Cov Model Before Training: Latent Length Scales",
    file_name="latent-length-scales-pre",
    file_path="./plot/model/mean-cov/parameter/",
    ymax=F.softplus(mean_cov_lit_model.full_model.cov_model.length_scales).detach().max().item(),
)

lambda_matrix_trained = mean_cov_lit_model.full_model.cov_model.lambda_matrix.unsqueeze(0)
cov_tensor_trained = mean_cov_lit_model.full_model.cov_model.build_covariance_matrix(lambda_matrix_trained)
cov_matrix_trained = (cov_tensor_trained @ cov_tensor_trained.transpose(-1, -2)).squeeze(0).detach().cpu().numpy()

for time_indices, time_name in zip([None, np.arange(15, 21), [2]], ["all", "15-20", "18"]):
    plot_covariance_matrix(
        mean_cov_lit_model=mean_cov_lit_model_pre,
        title="Mean-Cov Model Before Training: Covariance Matrix",
        file_name=f"covariance-matrix-pre-{time_name}",
        file_path="./plot/model/mean-cov/parameter/",
        time_indices=time_indices,
        bin_times=bin_times,
        vmin=-np.max(np.abs(cov_matrix_trained)),
        vmax=np.max(np.abs(cov_matrix_trained)),
    )


plot_cov_loading_matrix(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Mean-Cov Model: Latent Covariance Loadings",
    file_name="latent-covariance-loadings",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
)

plot_cov_noise(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Mean-Cov Model: Noise Variance",
    file_name="noise-variance",
    file_path="./plot/model/mean-cov/parameter/",
    bin_times=bin_times,
    unit_names=unit_names,
)

plot_cov_length_scales(
    mean_cov_lit_model=mean_cov_lit_model,
    title="Mean-Cov Model: Latent Length Scales",
    file_name="latent-length-scales",
    file_path="./plot/model/mean-cov/parameter/",
)

for time_indices, time_name in zip([None, np.arange(15, 21), [2]], ["all", "15-20", "18"]):
    plot_covariance_matrix(
        mean_cov_lit_model=mean_cov_lit_model,
        title="Mean-Cov Model: Covariance Matrix",
        file_name=f"covariance-matrix-{time_name}",
        file_path="./plot/model/mean-cov/parameter/",
        time_indices=time_indices,
        bin_times=bin_times,
    )

In [24]:
Y_train_baseline, Y_train_baseline_hat = predict_loader(Conf, baseline_lit_model, train_loader, Y_mean, variant="mean")
Y_valid_baseline, Y_valid_baseline_hat = predict_loader(Conf, baseline_lit_model, valid_loader, Y_mean, variant="mean")

Y_train_mean, Y_train_mean_hat = predict_loader(Conf, mean_lit_model, train_loader, Y_mean, variant="mean")
Y_valid_mean, Y_valid_mean_hat = predict_loader(Conf, mean_lit_model, valid_loader, Y_mean, variant="mean")

Y_train_mean_cov, Y_train_mean_cov_hat = predict_loader(Conf, mean_cov_lit_model, train_loader, Y_mean, variant="past_and_others")
Y_valid_mean_cov, Y_valid_mean_cov_hat = predict_loader(Conf, mean_cov_lit_model, valid_loader, Y_mean, variant="past_and_others")

In [25]:
unit_indices = [1, 23, 46]
trial_indices = [11, 23]


for unit_idx in unit_indices:
    for trial_idx in trial_indices:

        plot_prediction(
            Y=Y_valid_baseline,
            Y_hat=Y_valid_baseline_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Baseline Model: Unit {unit_names[unit_idx]}, trial {trial_idx} prediction",
            filename=f"trial_{trial_idx}_unit_{unit_idx}",
            filepath="./plot/model/baseline/prediction/",
        )

        plot_prediction(
            Y=Y_valid_mean,
            Y_hat=Y_valid_mean_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Mean Model: Unit {unit_names[unit_idx]}, trial {trial_idx} prediction",
            filename=f"trial_{trial_idx}_unit_{unit_idx}",
            filepath="./plot/model/mean/prediction/",
        )

        plot_prediction(
            Y=Y_valid_mean_cov,
            Y_hat=Y_valid_mean_cov_hat,
            trial_idx=trial_idx,
            unit_idx=unit_idx,
            bin_times=bin_times,
            title=f"Mean-Cov Model: Unit {unit_names[unit_idx]}, trial {trial_idx} prediction",
            filename=f"trial_{trial_idx}_unit_{unit_idx}",
            filepath="./plot/model/mean-cov/prediction/",
        )

In [26]:
correlation_trial_train_baseline, r2_trial_train_baseline, mse_trial_train_baseline = compute_metrics(Conf, Y_train_baseline, Y_train_baseline_hat)
correlation_unit_train_baseline = np.nanmean(correlation_trial_train_baseline, axis=1)
r2_unit_train_baseline = np.nanmean(r2_trial_train_baseline, axis=1)
mse_unit_train_baseline = np.nanmean(mse_trial_train_baseline, axis=1)

correlation_trial_valid_baseline, r2_trial_valid_baseline, mse_trial_valid_baseline = compute_metrics(Conf, Y_valid_baseline, Y_valid_baseline_hat)
correlation_unit_valid_baseline = np.nanmean(correlation_trial_valid_baseline, axis=1)
r2_unit_valid_baseline = np.nanmean(r2_trial_valid_baseline, axis=1)
mse_unit_valid_baseline = np.nanmean(mse_trial_valid_baseline, axis=1)


correlation_trial_train_mean, r2_trial_train_mean, mse_trial_train_mean = compute_metrics(Conf, Y_train_mean, Y_train_mean_hat)
correlation_unit_train_mean = np.nanmean(correlation_trial_train_mean, axis=1)
r2_unit_train_mean = np.nanmean(r2_trial_train_mean, axis=1)
mse_unit_train_mean = np.nanmean(mse_trial_train_mean, axis=1)

correlation_trial_valid_mean, r2_trial_valid_mean, mse_trial_valid_mean = compute_metrics(Conf, Y_valid_mean, Y_valid_mean_hat)
correlation_unit_valid_mean = np.nanmean(correlation_trial_valid_mean, axis=1)
r2_unit_valid_mean = np.nanmean(r2_trial_valid_mean, axis=1)
mse_unit_valid_mean = np.nanmean(mse_trial_valid_mean, axis=1)


correlation_trial_train_mean_cov, r2_trial_train_mean_cov, mse_trial_train_mean_cov = compute_metrics(Conf, Y_train_mean_cov, Y_train_mean_cov_hat)
correlation_unit_train_mean_cov = np.nanmean(correlation_trial_train_mean_cov, axis=1)
r2_unit_train_mean_cov = np.nanmean(r2_trial_train_mean_cov, axis=1)
mse_unit_train_mean_cov = np.nanmean(mse_trial_train_mean_cov, axis=1)

correlation_trial_valid_mean_cov, r2_trial_valid_mean_cov, mse_trial_valid_mean_cov = compute_metrics(Conf, Y_valid_mean_cov, Y_valid_mean_cov_hat)
correlation_unit_valid_mean_cov = np.nanmean(correlation_trial_valid_mean_cov, axis=1)
r2_unit_valid_mean_cov = np.nanmean(r2_trial_valid_mean_cov, axis=1)
mse_unit_valid_mean_cov = np.nanmean(mse_trial_valid_mean_cov, axis=1)

In [27]:
plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_baseline,
    correlations_valid=correlation_unit_valid_baseline,
    r2s_train=r2_unit_train_baseline,
    r2s_valid=r2_unit_valid_baseline,
    mses_train=mse_unit_train_baseline,
    mses_valid=mse_unit_valid_baseline,
    title="Baseline Model: Per-Unit Training and Validation Metrics",
    filename="metrics",
    filepath="./plot/model/baseline/",
)

plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_mean,
    correlations_valid=correlation_unit_valid_mean,
    r2s_train=r2_unit_train_mean,
    r2s_valid=r2_unit_valid_mean,
    mses_train=mse_unit_train_mean,
    mses_valid=mse_unit_valid_mean,
    title="Mean-Covariance Model: Per-Unit Training and Validation Metrics",
    filename="metrics",
    filepath="./plot/model/mean/",
)

plot_train_valid_metrics_comparison(
    correlations_train=correlation_unit_train_mean_cov,
    correlations_valid=correlation_unit_valid_mean_cov,
    r2s_train=r2_unit_train_mean_cov,
    r2s_valid=r2_unit_valid_mean_cov,
    mses_train=mse_unit_train_mean_cov,
    mses_valid=mse_unit_valid_mean_cov,
    title="Mean-Covariance Model: Per-Unit Training and Validation Metrics",
    filename="metrics",
    filepath="./plot/model/mean-cov/",
)

WindowsPath('plot/model/mean-cov')

In [28]:
plot_model_metrics_comparison(
    correlations_x_train=correlation_unit_train_baseline,
    correlations_x_valid=correlation_unit_valid_baseline,
    r2s_x_train=r2_unit_train_baseline,
    r2s_x_valid=r2_unit_valid_baseline,
    mses_x_train=mse_unit_train_baseline,
    mses_x_valid=mse_unit_valid_baseline,
    correlations_y_train=correlation_unit_train_mean,
    correlations_y_valid=correlation_unit_valid_mean,
    r2s_y_train=r2_unit_train_mean,
    r2s_y_valid=r2_unit_valid_mean,
    mses_y_train=mse_unit_train_mean,
    mses_y_valid=mse_unit_valid_mean,
    x_label="Baseline model",
    y_label="Mean model",
    title="Per-Unit Comparison: Baseline and Mean Models",
    file_name="baseline-vs-mean-metrics",
    file_path="./plot/model/comparison/",
)

plot_model_metrics_comparison(
    correlations_x_train=correlation_unit_train_mean,
    correlations_x_valid=correlation_unit_valid_mean,
    r2s_x_train=r2_unit_train_mean,
    r2s_x_valid=r2_unit_valid_mean,
    mses_x_train=mse_unit_train_mean,
    mses_x_valid=mse_unit_valid_mean,
    correlations_y_train=correlation_unit_train_mean_cov,
    correlations_y_valid=correlation_unit_valid_mean_cov,
    r2s_y_train=r2_unit_train_mean_cov,
    r2s_y_valid=r2_unit_valid_mean_cov,
    mses_y_train=mse_unit_train_mean_cov,
    mses_y_valid=mse_unit_valid_mean_cov,
    x_label="Mean model",
    y_label="Mean-Cov model",
    title="Per-Unit Comparison: Mean and Mean-Cov Models",
    file_name="mean-vs-mean-cov-metrics",
    file_path="./plot/model/comparison/",
)

WindowsPath('plot/model/comparison')

In [29]:
plot_model_metric_improvement(
    mse_trial_baseline=mse_trial_valid_baseline, 
    mse_trial_model=mse_trial_valid_mean,
    corr_trial_baseline=correlation_trial_valid_baseline, 
    corr_trial_model=correlation_trial_valid_mean,
    r2_trial_baseline=r2_trial_valid_baseline, 
    r2_trial_model=r2_trial_valid_mean,
    title="Baseline vs Mean Model Significant Improvements",
    file_name="baseline-vs-mean-improvement",
    file_path="./plot/model/comparison/",
)

plot_model_metric_improvement(
    mse_trial_baseline=mse_trial_valid_mean,
    mse_trial_model=mse_trial_valid_mean_cov,
    corr_trial_baseline=correlation_trial_valid_mean,
    corr_trial_model=correlation_trial_valid_mean_cov,
    r2_trial_baseline=r2_trial_valid_mean,
    r2_trial_model=r2_trial_valid_mean_cov,
    title="Mean vs Model Significant Improvements",
    file_name="mean-vs-model-improvement",
    file_path="./plot/model/comparison/",
)

WindowsPath('plot/model/comparison')

In [30]:
background, explain, shap_values, base_values = compute_shap_values(Conf, mean_lit_model, train_dataset, valid_dataset)

In [31]:
unit_bin_indices = [(1, 18), (23, 18), (46, 18)]
trial_indices = [11, 23]


for unit_idx, bin_idx in unit_bin_indices:
    shap_values_selected = shap_values[:, :, bin_idx, unit_idx]
    base_values_selected = base_values[:, bin_idx, unit_idx]


    data = np.concatenate([explain[0][:, bin_idx], explain[1], (explain[2] > 0).astype(int)], axis=1)


    explanation = shap.Explanation(values=shap_values_selected, base_values=base_values_selected, data=data, feature_names=variable_names)


    file_path = f"./plot/model/mean/shap/unit-bin/unit-{unit_idx:03d}-bin-{bin_idx:03d}/"

    for trial_idx in trial_indices:
        save_shap_plot(
            lambda: shap.plots.waterfall(explanation[trial_idx], max_display=len(variable_names), show=False),
            title=f"SHAP attribution — Unit {unit_names[unit_idx]}, output bin {bin_times[bin_idx]} s, trial {trial_idx}",
            file_name=f"trial-{trial_idx:03d}",
            file_path=f"{file_path}waterfall/",
        )

    for variable_name in variable_names:
        save_shap_plot(
            lambda: shap.plots.scatter(explanation[:, variable_name], show=False), 
            title=f"SHAP value versus variable-{variable_name} — Unit {unit_names[unit_idx]}, output bin {bin_times[bin_idx]} s", 
            file_name=f"variable-{variable_name}", 
            file_path=f"{file_path}scatter/",
        )

    save_shap_plot(
        lambda: shap.plots.beeswarm(explanation, max_display=len(variable_names), show=False), 
        title=f"Distribution of SHAP attributions across trials — Unit {unit_names[unit_idx]}, output bin {bin_times[bin_idx]} s", 
        file_name=f"beeswarm", 
        file_path=f"{file_path}summary/",
    )

    save_shap_plot(
        lambda: shap.plots.bar(explanation, max_display=len(variable_names), show=False), 
        title=f"Mean absolute SHAP attribution across trials — Unit {unit_names[unit_idx]}, output bin {bin_times[bin_idx]} s", 
        file_name=f"bar", 
        file_path=f"{file_path}summary/",
    )

    save_shap_plot(
        lambda: shap.plots.heatmap(explanation, show=False), 
        title=f"SHAP attributions across trials — Unit {unit_names[unit_idx]}, output bin {bin_times[bin_idx]} s", 
        file_name=f"heatmap", 
        file_path=f"{file_path}summary/",
    )

for trial_idx in trial_indices:
    for variable_idx, variable_name in enumerate(variable_names):        
        plot_shap(
            shap_values=shap_values[trial_idx, variable_idx], 
            title=f"SHAP attribution map — variable-{variable_name}, trial {trial_idx}", 
            file_name=f"variable-{variable_name}", 
            file_path=f"./plot/model/mean/shap/trial/trial-{trial_idx:03d}/", 
            bin_times=bin_times, 
            unit_names=unit_names,
            cmap="RdBu_r", 
            show=False,
        )

position, dense, sparse = explain

x = np.concatenate((
    position.transpose(0, 2, 1),
    np.broadcast_to(dense[:, :, None], (*dense.shape, Conf.data.n_bins)),
    np.broadcast_to(sparse[:, :, None], (*sparse.shape, Conf.data.n_bins))
), axis=1)

x = x - x.mean(axis=0, keepdims=True)
s = shap_values - shap_values.mean(axis=0, keepdims=True)

shap_pearson = np.einsum("tvb,tvbu->vbu", x, s) / np.sqrt(
    np.sum(x ** 2, axis=0)[..., None] * np.sum(s ** 2, axis=0)
)

for variable_idx, variable_name in enumerate(variable_names):        
    plot_shap(
        shap_values=shap_pearson[variable_idx], 
        title=f"Pearson correlation between input value and SHAP attribution — variable-{variable_name}",
        file_name=f"variable-{variable_name}", 
        file_path=f"./plot/model/mean/shap/population/pearson-shap-input/", 
        bin_times=bin_times, 
        unit_names=unit_names,
        cmap="RdBu_r", 
        show=False,
    )

for variable_idx, variable_name in enumerate(variable_names):        
    plot_shap(
        shap_values=np.abs(shap_values[:, variable_idx]).mean(axis=0), 
        title=f"Mean absolute SHAP attribution across trials — variable-{variable_name}",
        file_name=f"variable-{variable_name}", 
        file_path=f"./plot/model/mean/shap/population/mean-absolute-shap/", 
        bin_times=bin_times, 
        unit_names=unit_names,
        cmap="magma", 
        show=False,
    )

In [49]:
Conf_shuffle = copy.deepcopy(Conf)
Conf_shuffle.training.patience = 5
Conf_shuffle.optimization.reduce = 2

n_shuffles = 100

shap_values_shuffles = []

for shuffle_idx in tqdm(range(n_shuffles), desc="Shuffles"):

    Conf_shuffle.seed = shuffle_idx

    train_dataset_shuffle = shuffle_dataset(Conf_shuffle, train_dataset)
    valid_dataset_shuffle = shuffle_dataset(Conf_shuffle, valid_dataset)
    train_loader_shuffle, valid_loader_shuffle, loader_generators_shuffle = prepare_loader(Conf_shuffle, train_dataset_shuffle, valid_dataset_shuffle)

    mean_trainer_shuffle, mean_lit_model_shuffle = build_lit_model(Conf_shuffle, loader_generators_shuffle, "conditional", "identity", enable_progress_bar_epoch=False)
    
    mean_trainer_shuffle.fit(mean_lit_model_shuffle, train_loader_shuffle, valid_loader_shuffle)
    
    background_shuffle, explain_shuffle, shap_values_shuffle, base_values_shuffle = compute_shap_values(Conf_shuffle, mean_lit_model_shuffle, train_dataset_shuffle, valid_dataset_shuffle)

    shap_values_shuffles.append(shap_values_shuffle)

shap_values_shuffles = np.stack(shap_values_shuffles, axis=0)

Shuffles:   0%|          | 0/100 [00:00<?, ?it/s]

In [50]:
neuron_classes = {
    "movement": {
        "variables": ["x", "y", "d", "motion"],
        "time_window": (-3000, 2000)
    },
    "reward_prediction": {
        "variables": ["tslp", "last_choice"],
        "time_window": (-3000, 0)
    },
    "reward_outcome": {
        "variables": ["rew"],
        "time_window": (0, 2000)
    },
    "action_planning": {
        "variables": ["tunp", "choice"],
        "time_window": (0, 2000)
    }
}

class_names = list(neuron_classes.keys())

unit_selectivities = np.zeros((len(neuron_classes), n_units))
unit_selectivities_shuffles = np.zeros((n_shuffles, len(neuron_classes), n_units))

for class_idx, class_name in enumerate(class_names):
    class_config = neuron_classes[class_name]
    variable_mask = np.isin(variable_names, class_config["variables"])
    time_mask = (
        (bin_times >= class_config["time_window"][0]) & 
        (bin_times <= class_config["time_window"][1])
    )

    class_shap = shap_values[:, variable_mask, :, :]
    class_shap = class_shap[:, :, time_mask, :]
    
    class_shap_shuffles = shap_values_shuffles[:, :, variable_mask, :, :]
    class_shap_shuffles = class_shap_shuffles[:, :, :, time_mask, :]

    unit_selectivities[class_idx, :] = np.nanmean(np.abs(class_shap), axis=(0, 1, 2))
    unit_selectivities_shuffles[:, class_idx, :] = np.nanmean(np.abs(class_shap_shuffles), axis=(1, 2, 3))

In [61]:
alpha = 0.05
minimum_standardized_null  = 5

n_classes, n_units = unit_selectivities.shape

selectivity_p = np.full((n_classes, n_units), np.nan)
selectivity_q = np.full((n_classes, n_units), np.nan)
selectivity_standardized_null = np.full((n_classes, n_units), np.nan)

for class_idx in range(n_classes):
    for unit_idx in range(n_units):
        real_value = unit_selectivities[class_idx, unit_idx]
        null_values = unit_selectivities_shuffles[:, class_idx, unit_idx]

        selectivity_p[class_idx, unit_idx] = (1 + np.sum(null_values >= real_value)) / (1 + null_values.size)

        selectivity_standardized_null[class_idx, unit_idx] = (
            np.log(real_value + 1e-12) -
            np.median(np.log(null_values + 1e-12))
        ) / (
            1.4826 * np.median(
                np.abs(
                    np.log(null_values + 1e-12) -
                    np.median(np.log(null_values + 1e-12))
                )
            ) + 1e-12
        )

_, selectivity_q, _, _ = multipletests(
    selectivity_p.ravel(),
    alpha=alpha,
    method="fdr_bh"
)

selectivity_q = selectivity_q.reshape(selectivity_p.shape)
selectivity_significant = (selectivity_q < alpha) & (selectivity_standardized_null >= minimum_standardized_null )

In [ ]:
for unit_idx in list(range(n_units)):
    plot_unit_selectivity_hist(
        unit_selectivity_shuffles=unit_selectivities_shuffles[:, :, unit_idx],
        unit_selectivity=unit_selectivities[:, unit_idx],
        class_names=class_names,
        unit_name=unit_names[unit_idx],
        title=f"Shuffle Null Distributions: Unit {unit_names[unit_idx]}",
        filename=f"unit-{unit_names[unit_idx]}-shuffle-histograms",
        file_path="./plot/selectivity/units",
        show=False
    )

plot_unit_selectivity_curves(
    unit_selectivities=selectivity_standardized_null,
    class_names=class_names,
    selectivity_significant=selectivity_significant,
    title="Class-Unit Selectivity Curves",
    filename="selectivity-curves",
    file_path="./plot/selectivity"
)

plot_unit_class_selectivity_matrix(
    unit_selectivities=selectivity_standardized_null,
    class_names=class_names,
    unit_names=unit_names,
    selectivity_significant=selectivity_significant,
    title="Class-Unit Selectivity Map",
    filename="selectivity-map",
    file_path="./plot/selectivity"
)

plot_unit_selectivity_pca(
    unit_selectivities=selectivity_standardized_null,
    class_names=class_names,
    selectivity_significant=selectivity_significant,
    title="Unit Selectivity PCA",
    filename="unit-selectivity-pca",
    file_path="./plot/selectivity"
)

WindowsPath('plot/selectivity')